## Lab 07 – pliki, serializacja i dataclasses

1. Tokenizacja tekstu i worek słów – `tokenize`, `hamlet_stats`
2. JSON i pickle – `save_json`, `load_json`, `save_pickle`, `load_pickle`
3. Bezpieczny zapis pliku – `atomic_write`
4. Rekordy z dataclasses – `Run`, `parse_run`, `load_runs`
5. Program z argumentami wiersza poleceń – `newton`, `build_parser`, `main`

### Przypomnienie: pliki

* [`pathlib.Path`](https://docs.python.org/3/library/pathlib.html) opisuje ścieżkę; `/` łączy jej części, a `read_text` / `write_text` czytają i zapisują cały plik naraz,
* `with path.open(...) as handle:` zamyka plik po wyjściu z bloku – także po wyjątku,
* przy plikach tekstowych zawsze podawaj `encoding="utf-8"` – domyślne kodowanie zależy od systemu,
* [`tempfile.TemporaryDirectory`](https://docs.python.org/3/library/tempfile.html#tempfile.TemporaryDirectory) tworzy katalog, który znika po wyjściu z `with` – w testach nie zaśmiecamy repozytorium.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    note = Path(directory) / "notatka.txt"
    note.write_text("Zażółć gęślą jaźń\n", encoding="utf-8")
    with note.open(encoding="utf-8") as handle:
        first_line = handle.readline()
    print(repr(first_line), handle.closed)
    print([p.name for p in Path(directory).iterdir()])

## Zadanie 1

W lab06 `BagOfWords` dzielił tekst tylko po białych znakach, więc `"Hamlet,"`, `"hamlet"` i `"HAMLET."` były trzema różnymi słowami. Plik `data/hamlet.txt` zawiera pełny tekst *Hamleta* (z [Project Gutenberg](https://www.gutenberg.org/ebooks/1524), domena publiczna).

a) Napisz funkcję `tokenize(text)`, która przy pomocy wyrażenia regularnego zwraca listę słów zapisanych małymi literami. Słowo to ciąg liter (także spoza alfabetu angielskiego, np. `é`); wszystko inne – spacje, interpunkcja, cyfry, apostrofy – rozdziela słowa.

<details>
<summary>Podpowiedź</summary>

`\w` dopasowuje litery, ale też cyfry i znak `_`. Klasa `[^\W\d_]` oznacza „znak słowa, który nie jest cyfrą ani podkreśleniem”, czyli literę.

</details>

In [ ]:
import re


def tokenize(text):
    """Zwraca listę słów (ciągów liter) z tekstu, zapisanych małymi literami"""
    pass

In [ ]:
print(tokenize("Hamlet, Prince of Denmark!") == ["hamlet", "prince", "of", "denmark"])
print(tokenize("ACT III. Scene I.") == ["act", "iii", "scene", "i"])
print(tokenize("Ophelia’s 2nd song") == ["ophelia", "s", "nd", "song"])
print(tokenize("café — naïve") == ["café", "naïve"])
print(tokenize("") == [])

b) Napisz funkcję `hamlet_stats(path)`, która wczytuje tekst z pliku i zwraca krotkę: (liczba wystąpień słowa `"hamlet"`, lista 10 najczęstszych słów od najczęstszego). Możesz użyć swojej klasy `BagOfWords` z lab06 (wklej ją do komórki poniżej) albo [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter).

In [ ]:
from collections import Counter


def hamlet_stats(path):
    """Zwraca (liczba wystąpień słowa "hamlet", lista 10 najczęstszych słów)"""
    pass

In [ ]:
count, top10 = hamlet_stats("data/hamlet.txt")

print(count == 473)
print(top10 == ["the", "and", "to", "of", "i", "a", "you", "my", "hamlet", "in"])

> **Do przemyślenia:** „hamlet” jest dziewiątym najczęstszym słowem w sztuce. Czy to znaczy, że postacie aż tyle razy wypowiadają jego imię?

<details>
<summary>Odpowiedź</summary>

Nie. To tekst dramatu: przed każdą kwestią stoi imię mówiącej postaci, np. `HAMLET.` – takich linii jest 361 z 473 wystąpień. Liczba słów mówi więc głównie o tym, jak często Hamlet zabiera głos. To częsta pułapka w analizie tekstu: zanim policzysz statystyki, sprawdź, co faktycznie jest w danych – tu trzeba by najpierw usunąć oznaczenia mówców.

</details>

## Zadanie 2

Żeby zapisać obiekt na dysku i później go odtworzyć, trzeba go **zserializować**, czyli zamienić na ciąg znaków lub bajtów. Biblioteka standardowa ma do tego m.in.:
* [`json`](https://docs.python.org/3/library/json.html) – zapis tekstowy, czytelny dla człowieka i dla innych języków programowania, ale obsługuje tylko podstawowe typy: słowniki, listy, napisy, liczby, `True`/`False`/`None`,
* [`pickle`](https://docs.python.org/3/library/pickle.html) – zapis binarny prawie dowolnego obiektu Pythona, czytelny tylko dla Pythona.

Napisz cztery funkcje:
* `save_json(data, path)` i `load_json(path)` – zapis i odczyt przez `json` (plik tekstowy, UTF-8),
* `save_pickle(obj, path)` i `load_pickle(path)` – zapis i odczyt przez `pickle` (plik otwarty w trybie binarnym).

Funkcje nie powinny łapać wyjątków – jeśli danych nie da się zapisać, błąd ma dotrzeć do wywołującego.

In [ ]:
import json
import pickle


def save_json(data, path):
    """Zapisuje data do pliku path w formacie JSON"""
    pass


def load_json(path):
    """Wczytuje dane z pliku JSON"""
    pass


def save_pickle(obj, path):
    """Zapisuje obiekt do pliku path przez pickle"""
    pass


def load_pickle(path):
    """Wczytuje obiekt z pliku zapisanego przez pickle"""
    pass

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

counts = {"ala": 3, "ma": 2, "kota": 1}
data = {"pair": (1, 2), "tags": {"a", "b"}}

with TemporaryDirectory() as directory:
    directory = Path(directory)

    save_json(counts, directory / "counts.json")
    print(load_json(directory / "counts.json") == counts)
    print(directory.joinpath("counts.json").read_text(encoding="utf-8").startswith("{"))  # JSON to zwykły tekst

    save_pickle(counts, directory / "counts.pkl")
    print(load_pickle(directory / "counts.pkl") == counts)

    save_pickle(data, directory / "data.pkl")
    print(load_pickle(directory / "data.pkl") == data)  # pickle zachowuje krotki i zbiory

    try:
        save_json(data, directory / "data.json")  # zbioru nie da się zapisać w JSON
        print(False)
    except TypeError:
        print(True)

Uruchom poniższą komórkę (korzysta z `tokenize` z zadania 1) i porównaj rozmiary plików z licznikami słów *Hamleta*:

In [ ]:
from collections import Counter

hamlet_counts = dict(Counter(tokenize(Path("data/hamlet.txt").read_text(encoding="utf-8"))))

with TemporaryDirectory() as directory:
    directory = Path(directory)
    save_json(hamlet_counts, directory / "hamlet.json")
    save_pickle(hamlet_counts, directory / "hamlet.pkl")
    for name in ["hamlet.json", "hamlet.pkl"]:
        print(f"{name:12} {directory.joinpath(name).stat().st_size:>7} B")

> **Do przemyślenia:** pickle jest wygodniejszy (zapisze prawie wszystko) i tu nawet daje mniejszy plik. Dlaczego mimo to do wymiany danych częściej używa się JSON?

<details>
<summary>Odpowiedź</summary>

Przy domyślnych ustawieniach plik pickle wychodzi tu o ok. 15% mniejszy (ok. 51 kB wobec 61 kB). Mimo to:
* **bezpieczeństwo** – wczytanie pliku pickle może wykonać dowolny kod (obiekt sam opisuje, jak się odtworzyć). Nigdy nie wczytuj przez `pickle` danych z niezaufanego źródła. `json.load` zwraca tylko dane,
* **przenośność** – JSON przeczyta każdy język i człowiek; pickle tylko Python, czasem tylko podobna wersja i tylko gdy te same klasy są dostępne przy odczycie,
* **czytelność** – plik JSON da się otworzyć w edytorze i poprawić ręcznie.

pickle ma sens do tymczasowego zapisu własnych obiektów Pythona (np. cache wyników), JSON – do konfiguracji i wymiany danych.

</details>

<details>
<summary><b>Ciekawostka:</b> JSON po cichu zmienia typy</summary>

JSON ma tylko jeden rodzaj klucza (napis) i jeden rodzaj sekwencji (lista). Zapis i odczyt nie zawsze zwraca więc to samo:

```python
import json

print(json.loads(json.dumps({1: "a", 2: "b"})))
print(json.loads(json.dumps({"pair": (1, 2)})))
```
```
{'1': 'a', '2': 'b'}
{'pair': [1, 2]}
```

Klucze liczbowe stały się napisami, a krotka – listą. Zbioru nie da się zapisać wcale (`TypeError`), co widać w testach powyżej.

</details>

## Zadanie 3

Na wykładzie widzieliśmy, że `with` zamyka plik, ale nie cofa zapisu: jeśli program przerwie się w połowie zapisu, w pliku zostanie połowa danych, a poprzednia zawartość przepadnie. Typowe rozwiązanie to **zapis atomowy**: dane trafiają najpierw do pliku tymczasowego, a dopiero na koniec plik tymczasowy zastępuje docelowy.

Napisz menedżer kontekstu `atomic_write(path)` (przy pomocy [`contextlib.contextmanager`](https://docs.python.org/3/library/contextlib.html#contextlib.contextmanager)), używany tak:

```python
with atomic_write("wynik.txt") as handle:
    handle.write("nowa zawartość")
```

Wymagania:
* `handle` to plik tekstowy (UTF-8) – plik tymczasowy w **tym samym katalogu** co `path`,
* po udanym wyjściu z bloku plik tymczasowy zastępuje `path` (funkcja [`os.replace`](https://docs.python.org/3/library/os.html#os.replace) robi to atomowo),
* jeśli w bloku wystąpi wyjątek, `path` zostaje nietknięty, plik tymczasowy jest usuwany, a wyjątek leci dalej,
* do momentu wyjścia z bloku `path` ma starą zawartość.

<details>
<summary>Podpowiedź</summary>

Plik tymczasowy w wybranym katalogu, który nie zniknie sam po zamknięciu, utworzysz przez `tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=..., delete=False)`. Pamiętaj, żeby zamknąć go przed `os.replace`.

</details>

In [ ]:
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def atomic_write(path):
    """Zapisuje plik tekstowy atomowo: albo cała nowa zawartość, albo stara bez zmian"""
    pass

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    target = Path(directory) / "wynik.txt"
    target.write_text("stara", encoding="utf-8")

    with atomic_write(target) as handle:
        handle.write("nowa")
        print(target.read_text(encoding="utf-8") == "stara")  # przed wyjściem z bloku - bez zmian
    print(target.read_text(encoding="utf-8") == "nowa")

    try:
        with atomic_write(target) as handle:
            handle.write("tylko poło")
            raise RuntimeError("przerwany zapis")
        print(False)
    except RuntimeError:
        print(True)
    print(target.read_text(encoding="utf-8") == "nowa")  # stara zawartość zachowana

    with atomic_write(Path(directory) / "nowy.txt") as handle:  # nowy plik też działa
        handle.write("zażółć")
    print(Path(directory, "nowy.txt").read_text(encoding="utf-8") == "zażółć")

    print(sorted(p.name for p in Path(directory).iterdir()) == ["nowy.txt", "wynik.txt"])  # bez plików tymczasowych

> **Do przemyślenia:** dlaczego plik tymczasowy ma powstać w tym samym katalogu, a nie np. w systemowym `/tmp`?

<details>
<summary>Odpowiedź</summary>

`os.replace` jest atomowe tylko w obrębie jednego systemu plików – wtedy zmienia jedynie wpis w katalogu, a inne programy widzą albo stary, albo nowy plik. Katalog systemowy dla plików tymczasowych często leży na innym systemie plików (np. w pamięci RAM). Przeniesienie pliku między systemami plików to w praktyce kopiowanie, które samo może zostać przerwane w połowie – i atomowość znika.

</details>

## Zadanie 4

[`dataclasses`](https://docs.python.org/3/library/dataclasses.html) pozwalają zamienić luźne krotki i słowniki w czytelne rekordy. Wróćmy do pliku `data/atoms.log` z lab06 – zamiast filtrować linie, zamienimy każdą na obiekt.

Napisz:
* klasę `Run` z dekoratorem `@dataclass`, z polami `number: int`, `element: str` i `problem: str | None = None`. Obiekty mają być **niemodyfikowalne** (`frozen=True`) i **porównywalne** (`order=True`), żeby dało się je sortować,
* funkcję `parse_run(line)`, która zamienia jedną linię logu na obiekt `Run` (np. `RUN 000106 COMPLETED. OUTPUT IN FILE fluorine.dat. 1 OVERFLOW WARNING.` → `Run(106, "fluorine", "1 OVERFLOW WARNING.")`; gdy po `.dat.` nic nie ma, `problem` to `None`),
* funkcję `load_runs(path)`, która zwraca listę obiektów `Run` dla wszystkich linii pliku.

In [ ]:
import re
from dataclasses import dataclass


@dataclass
class Run:
    """Jeden wpis z data/atoms.log"""
    pass


def parse_run(line):
    """Zamienia linię logu na obiekt Run"""
    pass


def load_runs(path):
    """Zwraca listę obiektów Run dla wszystkich linii pliku"""
    pass

In [ ]:
import json
from dataclasses import FrozenInstanceError, asdict

print(parse_run("RUN 000106 COMPLETED. OUTPUT IN FILE fluorine.dat. 1 OVERFLOW WARNING.") == Run(106, "fluorine", "1 OVERFLOW WARNING."))
print(parse_run("RUN 000003 COMPLETED. OUTPUT IN FILE zinc.dat.") == Run(3, "zinc"))

runs = load_runs("data/atoms.log")
print(len(runs) == 40)
print(sum(run.problem is not None for run in runs) == 8)
print(sorted(runs)[0] == Run(3, "zinc"))  # order=True: sortowanie najpierw po number
print(sorted(runs)[-1].number == max(run.number for run in runs))
print(len({*runs, *runs}) == 40)  # frozen=True + eq=True: obiekty są hashowalne

try:
    runs[0].number = 1
    print(False)
except FrozenInstanceError:
    print(True)

print(Run(**json.loads(json.dumps(asdict(runs[0])))) == runs[0])  # dataclass -> słownik -> JSON i z powrotem

> **Do przemyślenia:** `order=True` porównuje obiekty `Run` tak jak krotki: najpierw `number`, potem `element`, potem `problem`. Co by się zmieniło, gdyby pole `element` było zdefiniowane jako pierwsze?

<details>
<summary>Odpowiedź</summary>

Sortowanie odbywałoby się alfabetycznie po nazwie pierwiastka, a nie po numerze eksperymentu – **kolejność pól w klasie decyduje o porządku**. Jeśli porządek ma zależeć od czegoś innego niż kolejność pól, można wyłączyć pole z porównań (`field(compare=False)`) albo sortować z jawnym kluczem, np. `sorted(runs, key=lambda run: run.element)`.

Dodatkowa pułapka: gdyby dwa wpisy miały ten sam `number` i `element`, porównanie doszłoby do `problem`, a `None < "tekst"` rzuca `TypeError`.

</details>

## Zadanie 5

Metoda Newtona szuka miejsca zerowego funkcji $f$: zaczyna od punktu $x_0$ i powtarza $x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$. Pochodną przybliżamy ilorazem różnicowym $f'(x) \approx \frac{f(x+h) - f(x)}{h}$, jak w lab04.

Na koniec zamienimy ją w program uruchamiany z terminala, np. `python newton.py "x**2 - 2" --x0 1`. Do zamiany tekstu wyrażenia na funkcję służy gotowa funkcja `make_function` – uruchom komórkę poniżej.

In [ ]:
import math


def make_function(expression):
    """Zamienia napis, np. "x**2 - 2" albo "cos(x) - x", na funkcję jednej zmiennej x.

    Uwaga: eval wykonuje kod z napisu - używaj tylko dla zaufanego wejścia.
    """
    names = {name: getattr(math, name) for name in dir(math) if not name.startswith("_")}
    code = compile(expression, "<expression>", "eval")
    return lambda x: eval(code, {"__builtins__": {}}, {**names, "x": x})


f = make_function("x**2 - 2")
print(f(3))

a) Napisz funkcję `newton(f, x0, h=1e-6, steps=50, eps=1e-10)`, która:
* zwraca `x`, gdy `abs(f(x)) < eps`,
* rzuca `ValueError`, jeśli po `steps` krokach nie znajdzie miejsca zerowego albo pochodna wyjdzie równa zero.

In [ ]:
def newton(f, x0, h=1e-6, steps=50, eps=1e-10):
    """Zwraca przybliżone miejsce zerowe funkcji f metodą Newtona, startując z x0"""
    pass

In [ ]:
import math

print(math.isclose(newton(lambda x: x ** 2 - 2, 1), math.sqrt(2)))
print(math.isclose(newton(lambda x: x ** 2 - 2, -1), -math.sqrt(2)))
print(math.isclose(newton(lambda x: math.cos(x) - x, 1), 0.7390851332, rel_tol=1e-9))

try:
    newton(lambda x: x ** 2 + x + 1, 0)  # brak pierwiastków rzeczywistych
    print(False)
except ValueError:
    print(True)

b) Napisz funkcję `build_parser()`, która zwraca [`argparse.ArgumentParser`](https://docs.python.org/3/library/argparse.html) z argumentami:
* `expression` – wyrażenie (argument pozycyjny),
* `--x0` – punkt startowy (`float`, domyślnie `1.0`),
* `--step` – krok `h` w pochodnej (`float`, domyślnie `1e-6`),
* `--steps` – maksymalna liczba kroków (`int`, domyślnie `50`),
* `--eps` – dokładność (`float`, domyślnie `1e-10`).

Dodaj krótkie opisy (`help=...`) – `argparse` sam zbuduje z nich pomoc dostępną pod `-h` / `--help`.

In [ ]:
import argparse


def build_parser():
    """Zwraca parser argumentów programu newton"""
    pass

In [ ]:
import contextlib
import io

parser = build_parser()

args = parser.parse_args(["x**2 - 2"])
print(args.expression == "x**2 - 2")
print((args.x0, args.step, args.steps, args.eps) == (1.0, 1e-6, 50, 1e-10))

args = parser.parse_args(["cos(x) - x", "--x0", "0.5", "--steps", "10"])
print(args.x0 == 0.5 and args.steps == 10 and isinstance(args.steps, int))

with contextlib.redirect_stderr(io.StringIO()):  # argparse wypisuje błąd na stderr
    try:
        parser.parse_args(["x", "--steps", "dużo"])
        print(False)
    except SystemExit:  # zły typ argumentu kończy program
        print(True)

print("--x0" in parser.format_help())

c) Napisz funkcję `main(argv=None)`, która parsuje argumenty z listy `argv` (`None` oznacza argumenty z terminala), buduje funkcję przez `make_function`, uruchamia `newton` z podanymi parametrami, wypisuje wynik i go zwraca.

In [ ]:
def main(argv=None):
    """Uruchamia program newton dla argumentów z listy argv"""
    pass

In [ ]:
import math

print(math.isclose(main(["x**2 - 2", "--x0", "1"]), math.sqrt(2)))
print(math.isclose(main(["cos(x) - x"]), 0.7390851332, rel_tol=1e-9))
print(math.isclose(main(["x**3 - 8", "--x0", "5", "--steps", "100"]), 2))

> **Do przemyślenia:** pierwotna wersja tego zadania podawała przykład `./newton.py x**2+x+1 -h 0.00001` (z `-h` jako krokiem pochodnej). Dlaczego to nie mogło działać – i to z dwóch powodów?

<details>
<summary>Odpowiedź</summary>

1. `argparse` sam dodaje opcję `-h` / `--help`. Próba zdefiniowania własnego `-h` kończy się błędem `conflicting option string`, a użytkownik i tak spodziewa się pod `-h` pomocy. Dlatego krok nazwaliśmy `--step`.
2. $x^2 + x + 1$ nie ma pierwiastków rzeczywistych (wyróżnik $1 - 4 < 0$), więc metoda Newtona nigdy nie zbiegnie – nasza funkcja rzuci `ValueError`.

Przykłady w zadaniach też trzeba testować.

</details>

<details>
<summary><b>Ciekawostka:</b> ten sam kod jako prawdziwy skrypt</summary>

Żeby uruchomić program z terminala, wystarczy zapisać funkcje w pliku i dodać na końcu:

```python
if __name__ == "__main__":
    main()
```

Blok wykona się tylko przy uruchomieniu pliku (`python newton.py ...`), a nie przy imporcie (`import newton`). `main()` bez argumentów czyta wtedy `sys.argv[1:]`:

```
$ python newton.py "x**2 - 2" --x0 1
1.4142135623754424
$ python newton.py --help
usage: newton.py [-h] [--x0 X0] [--step STEP] [--steps STEPS] [--eps EPS]
                 expression
...
```

W notebooku `main()` bez argumentów nie zadziała – `sys.argv` zawiera tam argumenty jądra Jupytera. Dlatego w testach przekazujemy listę jawnie.

</details>